# V11 — Unsupervised Clustering of Rope Flow Cycles

Clone repo, extract all cycles, cluster with K-means + HDBSCAN, visualize with t-SNE + UMAP.
No labels needed for clustering — labels used only for evaluation after.

In [ ]:
# ── Cell 1: Setup — clone repo + install deps ────────────────
!git clone https://github.com/helenejabbour/ropeflow-project.git 2>/dev/null || echo 'Already cloned'
!pip install hdbscan umap-learn -q

BASE = 'ropeflow-project'
DATA_PROCESSED = f'{BASE}/data/processed'
NEW_LABELED_RAW = f'{BASE}/data/raw/new-labeled-sessions'
RESULTS_DIR = 'v11_results'

import os
os.makedirs(RESULTS_DIR, exist_ok=True)
print(f'Data: {DATA_PROCESSED}')
print(f'Labels: {NEW_LABELED_RAW}')
print(f'Results: {RESULTS_DIR}')

In [ ]:
# ── Cell 2: Imports ───────────────────────────────────────────
import glob, re, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.signal import savgol_filter, find_peaks
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, adjusted_rand_score, normalized_mutual_info_score
from sklearn.manifold import TSNE
import hdbscan
import umap
from collections import Counter, defaultdict
import warnings
warnings.filterwarnings('ignore')
np.random.seed(42)
print('Imports OK')

In [ ]:
# ── Cell 3: Signal processing functions ──────────────────────

CONFIG = {
    'FS': 50.0,
    'CYCLE_PROMINENCE_DEGS': 100.0,
    'CYCLE_MIN_PERIOD_S': 0.5,
    'CYCLE_MAX_PERIOD_S': 3.0,
    'TARGET_LEN': 64,
    'MIN_CYCLE_SAMPLES': 10,
}

def extract_signals(df):
    t = df['timestamp_ms'].values / 1000.0
    A = df[['ax_w', 'ay_w', 'az_w']].values
    omega = df[['gx', 'gy', 'gz']].values * (np.pi / 180.0)
    return t, A, omega

def detect_cycles(t, omega, fs=50.0):
    mag = np.linalg.norm(omega, axis=1)
    mag_smooth = savgol_filter(mag, window_length=15, polyorder=3)
    prom = CONFIG['CYCLE_PROMINENCE_DEGS'] * np.pi / 180.0
    min_dist = int(CONFIG['CYCLE_MIN_PERIOD_S'] * fs)
    peaks, _ = find_peaks(mag_smooth, distance=min_dist, prominence=prom)
    if len(peaks) < 2:
        return []
    bounds = [0] + [(peaks[i]+peaks[i+1])//2 for i in range(len(peaks)-1)] + [len(t)-1]
    return [(bounds[i], bounds[i+1]) for i in range(len(bounds)-1)
            if CONFIG['CYCLE_MIN_PERIOD_S'] <= t[bounds[i+1]]-t[bounds[i]] <= CONFIG['CYCLE_MAX_PERIOD_S']
            and (bounds[i+1]-bounds[i]) >= CONFIG['MIN_CYCLE_SAMPLES']]

def pair_cycles(t0, cyc0, t1, cyc1):
    paired0, paired1, used = [], [], set()
    for c0 in cyc0:
        best_i, best_ov = -1, -1.0
        for i, c1 in enumerate(cyc1):
            if i in used: continue
            ov = max(0, min(t0[c0[1]], t1[c1[1]]) - max(t0[c0[0]], t1[c1[0]]))
            if ov > best_ov: best_ov, best_i = ov, i
        if best_i >= 0 and best_ov > 0:
            paired0.append(c0); paired1.append(cyc1[best_i]); used.add(best_i)
    return paired0, paired1

def resample(sig, n):
    if len(sig) < 2:
        return np.zeros(n) if sig.ndim == 1 else np.zeros((n, sig.shape[1]))
    x_old, x_new = np.linspace(0, 1, len(sig)), np.linspace(0, 1, n)
    if sig.ndim == 1: return np.interp(x_new, x_old, sig)
    return np.column_stack([np.interp(x_new, x_old, sig[:, j]) for j in range(sig.shape[1])])

def build_cycle_matrix(A0, A1, om0, om1, s0, e0, s1, e1):
    tl = CONFIG['TARGET_LEN']
    r0 = resample(np.column_stack([A0[s0:e0], om0[s0:e0]]), tl)
    r1 = resample(np.column_stack([A1[s1:e1], om1[s1:e1]]), tl)
    return np.column_stack([r0, r1]).T  # (12, 64)

# Label mapping
_EXACT = {
    'underhand': 'underhand', 'overhand': 'overhand', 'dragon_roll': 'dragon_roll',
    'sneak_underhand': 'sneak_underhand', 'sneak_overhand': 'sneak_overhand', 'idle': None,
}
_PREFIX_RULES = [
    (re.compile(r'^us', re.I), 'sneak_underhand'), (re.compile(r'^os', re.I), 'sneak_overhand'),
    (re.compile(r'^u', re.I), 'underhand'), (re.compile(r'^o', re.I), 'overhand'),
    (re.compile(r'^fb', re.I), 'dragon_roll'), (re.compile(r'^bf', re.I), 'dragon_roll'),
    (re.compile(r'^cw$', re.I), 'CW'), (re.compile(r'^ccw$', re.I), 'CCW'),
    (re.compile(r'^idle', re.I), None), (re.compile(r'^vq', re.I), None),
]
KNOWN_PATTERNS = {'overhand', 'sneak_overhand', 'underhand', 'sneak_underhand',
                  'dragon_roll', 'underhand_default'}

def map_label(raw):
    raw = raw.strip()
    if raw.lower() in _EXACT: return _EXACT[raw.lower()]
    for pat, c in _PREFIX_RULES:
        if pat.match(raw): return c
    return None

print('Functions defined')

In [ ]:
# ── Cell 4: Load ALL cycles from ALL sessions ───────────────

all_matrices = []
all_labels = []
all_session_ids = []
all_cycle_times = []
session_names = []

def process_session_file(d0_path, d1_path, session_name, label='unlabeled', time_windows=None):
    df0 = pd.read_csv(d0_path)
    df1 = pd.read_csv(d1_path)
    t0, A0, om0 = extract_signals(df0)
    t1, A1, om1 = extract_signals(df1)
    fs = CONFIG['FS']
    cyc0 = detect_cycles(t0, om0, fs)
    cyc1 = detect_cycles(t1, om1, fs)
    p0, p1 = pair_cycles(t0, cyc0, t1, cyc1)
    if time_windows is not None:
        fp0, fp1 = [], []
        for (s0, e0), (s1, e1) in zip(p0, p1):
            t_mid = (t0[s0] + t0[e0]) / 2
            if any(ws <= t_mid < we for ws, we in time_windows):
                fp0.append((s0, e0)); fp1.append((s1, e1))
        p0, p1 = fp0, fp1
    sid = len(session_names)
    session_names.append(session_name)
    count = 0
    for (s0, e0), (s1, e1) in zip(p0, p1):
        cm = build_cycle_matrix(A0, A1, om0, om1, s0, e0, s1, e1)
        all_matrices.append(cm)
        all_labels.append(label)
        all_session_ids.append(sid)
        all_cycle_times.append((t0[s0] + t0[e0]) / 2)
        count += 1
    return count

# Homogeneous sessions
d0_files = sorted(glob.glob(os.path.join(DATA_PROCESSED, '*_device0_processed.csv')))
for d0 in d0_files:
    d1 = d0.replace('_device0_', '_device1_')
    if not os.path.exists(d1): continue
    stem = os.path.basename(d0).replace('_device0_processed.csv', '')
    parts = stem.split('_')
    if len(parts) < 3: continue
    rest = '_'.join(parts[2:])
    label = 'unlabeled'
    for pat in sorted(KNOWN_PATTERNS, key=len, reverse=True):
        if rest.startswith(pat):
            label = 'underhand' if pat == 'underhand_default' else pat
            break
    n = process_session_file(d0, d1, stem, label)
    print(f'  {stem:<55s} {label:<20s} {n:>4d} cycles')

# Heterogeneous sessions
if os.path.isdir(NEW_LABELED_RAW):
    for sname in sorted(os.listdir(NEW_LABELED_RAW)):
        sdir = os.path.join(NEW_LABELED_RAW, sname)
        if not os.path.isdir(sdir): continue
        lpath = None
        for fn in ('labels_corrected.json', 'labels.json'):
            p = os.path.join(sdir, fn)
            if os.path.isfile(p): lpath = p; break
        if not lpath: continue
        d0 = os.path.join(DATA_PROCESSED, f'{sname}_device0_processed.csv')
        d1 = os.path.join(DATA_PROCESSED, f'{sname}_device1_processed.csv')
        if not (os.path.isfile(d0) and os.path.isfile(d1)): continue
        with open(lpath, encoding='utf-8') as f:
            data = json.load(f)
        segs = data.get('segments', data) if isinstance(data, dict) else data
        wbl = defaultdict(list)
        for seg in segs:
            canon = map_label(seg.get('label', ''))
            if canon is None: continue
            s, e = seg.get('start'), seg.get('end')
            if s is None: continue
            if e is None: e = s + 2.0
            wbl[canon].append((float(s), float(e)))
        for label, windows in sorted(wbl.items()):
            n = process_session_file(d0, d1, f'{sname}/{label}', label, windows)
            if n > 0:
                print(f'  {sname}/{label:<20s} {"":>34s} {n:>4d} cycles')

X_all = np.array([cm.ravel() for cm in all_matrices])
labels_all = np.array(all_labels)

print(f'\nTotal: {len(X_all)} cycles, {X_all.shape[1]}D')
print(f'Label distribution:')
for lab, cnt in sorted(Counter(labels_all).items(), key=lambda x: -x[1]):
    print(f'  {lab:<20s}: {cnt:>5d}')

In [ ]:
# ── Cell 5: PCA reduction + t-SNE/UMAP ───────────────────────
from sklearn.decomposition import PCA

scaler = StandardScaler()
X_scaled_768 = scaler.fit_transform(X_all)

# PCA: 768D -> keep 95% variance (matches Helene's V10 approach)
pca = PCA(n_components=0.95, svd_solver='full')
X_pca = pca.fit_transform(X_scaled_768)
n_pca = X_pca.shape[1]
cumvar = np.cumsum(pca.explained_variance_ratio_)
print(f'PCA: 768D -> {n_pca}D (95% variance)')
print(f'  Explained variance: {cumvar[-1]*100:.1f}%')
print(f'  Top 5 components explain: {cumvar[4]*100:.1f}%')
print(f'  Top 10 components explain: {cumvar[min(9,n_pca-1)]*100:.1f}%')

# Plot explained variance curve
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(range(1, n_pca+1), cumvar*100, 'o-', color='#7F77DD', markersize=3)
ax.axhline(95, color='red', ls='--', lw=1, label='95% threshold')
ax.axhline(99, color='orange', ls='--', lw=1, label='99% threshold')
ax.set_xlabel('Number of PCA components')
ax.set_ylabel('Cumulative explained variance (%)')
ax.set_title(f'PCA: {n_pca} components for 95% variance')
ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'fig_pca_variance.png'), dpi=150)
plt.show()

# Use PCA-reduced data for all downstream
X_scaled = X_pca

print('Running t-SNE on ' + str(n_pca) + 'D PCA space...')
tsne = TSNE(n_components=2, perplexity=30, random_state=42, n_iter=1000)
X_tsne = tsne.fit_transform(X_scaled)
print(f'  t-SNE done: {X_tsne.shape}')

print('Running UMAP on ' + str(n_pca) + 'D PCA space...')
reducer = umap.UMAP(n_components=2, n_neighbors=15, min_dist=0.1, random_state=42)
X_umap = reducer.fit_transform(X_scaled)
print(f'  UMAP done: {X_umap.shape}')

np.save(os.path.join(RESULTS_DIR, 'X_pca.npy'), X_pca)
np.save(os.path.join(RESULTS_DIR, 'X_tsne.npy'), X_tsne)
np.save(os.path.join(RESULTS_DIR, 'X_umap.npy'), X_umap)
np.save(os.path.join(RESULTS_DIR, 'labels_all.npy'), labels_all)
print('All embeddings saved')


In [ ]:
# ── Cell 6: K-means clustering (on PCA-reduced space) ────────

K_VALUES = [5, 8, 10, 11, 12, 13, 15]
kmeans_results = {}
inertias = []
silhouettes = []

labeled_mask = labels_all != 'unlabeled'

print('K-means clustering on ' + str(X_scaled.shape[1]) + 'D PCA space:')
for k in K_VALUES:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    cl = km.fit_predict(X_scaled)
    sil = silhouette_score(X_scaled, cl)
    inertias.append(km.inertia_)
    silhouettes.append(sil)
    kmeans_results[k] = cl
    if labeled_mask.sum() > 0:
        ari = adjusted_rand_score(labels_all[labeled_mask], cl[labeled_mask])
        nmi = normalized_mutual_info_score(labels_all[labeled_mask], cl[labeled_mask])
        print(f'  k={k:>2d}: silhouette={sil:.3f}, ARI={ari:.3f}, NMI={nmi:.3f}')
    else:
        print(f'  k={k:>2d}: silhouette={sil:.3f}')

best_k = K_VALUES[np.argmax(silhouettes)]
print(f'Best k by silhouette: {best_k} ({max(silhouettes):.3f})')


In [ ]:
# ── Cell 7: HDBSCAN clustering (on PCA-reduced space) ────────

clusterer = hdbscan.HDBSCAN(min_cluster_size=15, min_samples=5, metric='euclidean')
hdb_labels = clusterer.fit_predict(X_scaled)
n_clusters_hdb = len(set(hdb_labels)) - (1 if -1 in hdb_labels else 0)
n_noise = (hdb_labels == -1).sum()
print('HDBSCAN on ' + str(X_scaled.shape[1]) + 'D PCA: ' + str(n_clusters_hdb) + ' clusters, ' + str(n_noise) + ' noise (' + f'{n_noise/len(hdb_labels)*100:.1f}%)')

non_noise = hdb_labels != -1
if labeled_mask.sum() > 0 and (labeled_mask & non_noise).sum() > 10:
    mask = labeled_mask & non_noise
    ari = adjusted_rand_score(labels_all[mask], hdb_labels[mask])
    nmi = normalized_mutual_info_score(labels_all[mask], hdb_labels[mask])
    print(f'ARI={ari:.3f}, NMI={nmi:.3f} (labeled non-noise)')


In [ ]:
# ── Cell 8: Plot — embeddings colored by known labels ────────

LABEL_COLORS = {
    'underhand': '#5DCAA5', 'overhand': '#7F77DD', 'dragon_roll': '#E24B4A',
    'sneak_overhand': '#D85A30', 'sneak_underhand': '#EF9F27',
    'CW': '#3498db', 'CCW': '#9b59b6', 'unlabeled': '#DDDDDD',
}

fig, axes = plt.subplots(1, 2, figsize=(20, 8))
for ax, X_2d, method in zip(axes, [X_tsne, X_umap], ['t-SNE', 'UMAP']):
    mask_unl = labels_all == 'unlabeled'
    ax.scatter(X_2d[mask_unl, 0], X_2d[mask_unl, 1], c='#DDDDDD', s=5, alpha=0.3, label='unlabeled')
    for label in sorted(set(labels_all)):
        if label == 'unlabeled': continue
        mask = labels_all == label
        ax.scatter(X_2d[mask, 0], X_2d[mask, 1], c=LABEL_COLORS.get(label, '#888'),
                   s=15, alpha=0.7, label=label)
    ax.set_title(f'{method} colored by known labels', fontsize=13)
    ax.legend(fontsize=8, markerscale=2)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'fig_embeddings_by_label.png'), dpi=150)
plt.show()

In [ ]:
# ── Cell 9: Plot — elbow + silhouette ─────────────────────────

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
ax1.plot(K_VALUES, inertias, 'o-', color='#7F77DD', lw=2)
ax1.set_xlabel('k'); ax1.set_ylabel('Inertia'); ax1.set_title('Elbow Method')
ax1.set_xticks(K_VALUES)
ax2.plot(K_VALUES, silhouettes, 's-', color='#5DCAA5', lw=2)
ax2.set_xlabel('k'); ax2.set_ylabel('Silhouette Score'); ax2.set_title('Silhouette vs k')
ax2.set_xticks(K_VALUES)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'fig_elbow_silhouette.png'), dpi=150)
plt.show()

In [ ]:
# ── Cell 10: Plot — t-SNE colored by K-means (all k) ─────────

fig, axes = plt.subplots(2, 4, figsize=(24, 12))
for ax, k in zip(axes.flat[:len(K_VALUES)], K_VALUES):
    cl = kmeans_results[k]
    ax.scatter(X_tsne[:, 0], X_tsne[:, 1], c=cl, cmap='tab20', s=5, alpha=0.5)
    ax.set_title(f'k={k} (sil={silhouettes[K_VALUES.index(k)]:.3f})', fontsize=10)
if len(K_VALUES) < 8:
    axes.flat[len(K_VALUES)].axis('off')
plt.suptitle('t-SNE colored by K-means clusters', fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'fig_tsne_kmeans_grid.png'), dpi=150)
plt.show()

In [ ]:
# ── Cell 11: Plot — HDBSCAN on t-SNE and UMAP ────────────────

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(20, 8))
noise_mask = hdb_labels == -1
for ax, X_2d, method in zip([ax1, ax2], [X_tsne, X_umap], ['t-SNE', 'UMAP']):
    ax.scatter(X_2d[noise_mask, 0], X_2d[noise_mask, 1], c='#DDDDDD', s=3, alpha=0.3, label='noise')
    sc = ax.scatter(X_2d[~noise_mask, 0], X_2d[~noise_mask, 1],
                    c=hdb_labels[~noise_mask], cmap='tab20', s=10, alpha=0.6)
    ax.set_title(f'{method} + HDBSCAN ({n_clusters_hdb} clusters)', fontsize=13)
    plt.colorbar(sc, ax=ax, label='Cluster')
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'fig_hdbscan_embeddings.png'), dpi=150)
plt.show()

In [ ]:
# ── Cell 12: Cluster analysis — composition + purity ─────────

def analyze_clusters(cluster_labels, name):
    unique = sorted(set(cluster_labels))
    if -1 in unique: unique.remove(-1)
    print(f'\n{name}: {len(unique)} clusters')
    print(f'{"Cluster":>8s}  {"Size":>6s}  {"Dominant":>20s}  {"Purity":>8s}  Composition')
    print('-' * 90)
    for c in unique:
        mask = cluster_labels == c
        size = mask.sum()
        lc = Counter(labels_all[mask])
        lc_known = {k: v for k, v in lc.items() if k != 'unlabeled'}
        if lc_known:
            dom = max(lc_known, key=lc_known.get)
            pur = lc_known[dom] / sum(lc_known.values())
        else:
            dom = 'unlabeled'; pur = 0.0
        comp = ', '.join(f'{l}={c}' for l, c in sorted(lc.items(), key=lambda x: -x[1])[:5])
        print(f'  C{c:>3d}    {size:>5d}  {dom:>20s}  {pur:>7.1%}  {comp}')
    # Overall purity
    total_known = sum(1 for l in labels_all if l != 'unlabeled')
    if total_known > 0:
        correct = sum(max(Counter({k:v for k,v in Counter(labels_all[cluster_labels==c]).items() if k!='unlabeled'}).values(), default=0) for c in unique)
        print(f'Overall purity: {correct/total_known:.3f}')

for k in K_VALUES:
    analyze_clusters(kmeans_results[k], f'K-means k={k}')

analyze_clusters(hdb_labels, 'HDBSCAN')

In [ ]:
# ── Cell 13: Save results ─────────────────────────────────────

results_df = pd.DataFrame({
    'session': [session_names[sid] for sid in all_session_ids],
    'label': labels_all,
    'cycle_time': all_cycle_times,
    'hdbscan_cluster': hdb_labels,
})
for k in K_VALUES:
    results_df[f'kmeans_k{k}'] = kmeans_results[k]
results_df.to_csv(os.path.join(RESULTS_DIR, 'cluster_assignments.csv'), index=False)

print(f'\nSUMMARY')
print(f'Total cycles: {len(X_all)}')
print(f'Feature dim: {X_all.shape[1]}')
print(f'Labeled: {sum(labels_all != "unlabeled")}, Unlabeled: {sum(labels_all == "unlabeled")}')
print(f'\nK-means silhouette:')
for k, sil in zip(K_VALUES, silhouettes):
    marker = ' <-- best' if k == K_VALUES[np.argmax(silhouettes)] else ''
    print(f'  k={k:>2d}: {sil:.4f}{marker}')
print(f'\nHDBSCAN: {n_clusters_hdb} natural clusters')
print(f'\nSaved: {RESULTS_DIR}/')